## 1. Setup & Initialization

### Install Dependencies (if needed)

In [1]:
# Required packages (uncomment to install)
# !pip install langchain langchain-core langchain-postgres langchain-community langchain-huggingface sqlalchemy asyncpg psycopg2-binary nest-asyncio

# For DiskANN support, install pgvectorscale extension in PostgreSQL:
# https://github.com/timescale/pgvectorscale

In [2]:
import sys
import os

# Ensure the parent directory is in sys.path for module import
notebook_dir = os.path.dirname(os.path.abspath('demo.ipynb'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

### Import Required Libraries

In [3]:
import asyncio
import time
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
import nest_asyncio

# Import the production RAG system
from src.core import (
    pgVectorDB,
    IndexType,
    StorageLayout,
    DistanceMetric
)

# Allow nested event loops in Jupyter
nest_asyncio.apply()

print("✓ Imports successful")

✓ Imports successful


### Configure Database Connection

In [4]:
# PostgreSQL connection settings - UPDATE WITH YOUR CREDENTIALS
DB_HOST = "localhost"
DB_PORT = "9002"
DB_NAME = "postgres"
DB_USER = "user"
DB_PASSWORD = "root"

connection_string = f"postgresql+asyncpg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
print("✓ Connection string configured")

✓ Connection string configured


### Initialize Embedding Model

In [5]:
# Initialize embedding model (384 dimensions)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✓ Embedding model loaded")

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


✓ Embedding model loaded


## 2. Sample Documents & Labels

In [5]:
# Create diverse sample documents
documents = [
    Document(
        page_content="Python is a high-level programming language known for its simplicity and readability.",
        metadata={"category": "programming", "language": "Python", "year": 2024, "author": "Tech Expert"}
    ),
    Document(
        page_content="Machine learning algorithms can identify patterns in large datasets automatically.",
        metadata={"category": "ai", "language": "Python", "year": 2024, "author": "AI Researcher"}
    ),
    Document(
        page_content="PostgreSQL is a powerful open-source relational database management system.",
        metadata={"category": "database", "language": "SQL", "year": 2023, "author": "DB Expert"}
    ),
    Document(
        page_content="Natural language processing enables computers to understand and generate human language.",
        metadata={"category": "ai", "language": "Python", "year": 2024, "author": "NLP Specialist"}
    ),
    Document(
        page_content="React is a JavaScript library for building user interfaces with reusable components.",
        metadata={"category": "web", "language": "JavaScript", "year": 2023, "author": "Frontend Developer"}
    ),
    Document(
        page_content="Vector databases store and retrieve data based on semantic similarity using embeddings.",
        metadata={"category": "database", "language": "Python", "year": 2024, "author": "Data Engineer"}
    ),
    Document(
        page_content="Deep learning neural networks can solve complex problems like image recognition.",
        metadata={"category": "ai", "language": "Python", "year": 2024, "author": "Deep Learning Expert"}
    ),
    Document(
        page_content="FastAPI is a modern Python framework for building high-performance APIs quickly.",
        metadata={"category": "programming", "language": "Python", "year": 2023, "author": "Backend Developer"}
    ),
    Document(
        page_content="Docker containers provide isolated environments for running applications consistently.",
        metadata={"category": "devops", "language": "Shell", "year": 2023, "author": "DevOps Engineer"}
    ),
    Document(
        page_content="Transformer models revolutionized NLP with attention mechanisms and parallel processing.",
        metadata={"category": "ai", "language": "Python", "year": 2024, "author": "ML Researcher"}
    )
]

# Labels for DiskANN filtering
# 1=programming, 2=ai, 3=database, 4=web, 5=devops
document_labels = [
    [1],      # Python programming
    [2],      # Machine learning (ai)
    [3],      # PostgreSQL (database)
    [2],      # NLP (ai)
    [4],      # React (web)
    [3],      # Vector databases (database)
    [2],      # Deep learning (ai)
    [1],      # FastAPI (programming)
    [5],      # Docker (devops)
    [2],      # Transformers (ai)
]

print(f"✓ Prepared {len(documents)} documents with labels")
print("\nLabel mapping:")
print("  1 = programming")
print("  2 = ai")
print("  3 = database")
print("  4 = web")
print("  5 = devops")

✓ Prepared 10 documents with labels

Label mapping:
  1 = programming
  2 = ai
  3 = database
  4 = web
  5 = devops


## 3. HNSW Index Demo (Fast, In-Memory)

### Create & Initialize HNSW System

In [6]:
# Create RAG system with HNSW index
hnsw_rag = pgVectorDB(
    collection_name="hnsw_prod_demo",
    embedding_model=embedding_model,
    connection_string=connection_string,
    index_type=IndexType.HNSW
)

# Initialize system
await hnsw_rag.initialize(overwrite_existing=True)
print("✓ HNSW system initialized")

INFO:src.core:pgVectorDB initialized: 'hnsw_prod_demo' with hnsw (vector_size=384)
INFO:src.core:Step 1/5: Ensuring PostgreSQL extensions...
INFO:src.core:✓ Extension 'vector' enabled
INFO:src.core:✓ Extension 'pg_trgm' enabled
INFO:src.core:Step 2/5: Creating table...
INFO:src.core:✓ Table 'hnsw_prod_demo' created
INFO:src.core:Step 3/5: Setting up search indexes...
INFO:src.core:✓ Full-text search index created
INFO:src.core:✓ Trigram similarity index created
INFO:src.core:Step 4/5: Initializing vector store...
INFO:src.core:✓ Vector store initialized
INFO:src.core:✓ Step 5/5: System ready with hnsw index (vector_size=384)
INFO:src.core:================================================================================
INFO:src.core:🚀 Production RAG System initialized successfully!
INFO:src.core:   - Extensions: vector, pg_trgm
INFO:src.core:   - Table: public.hnsw_prod_demo
INFO:src.core:   - Indexes: Full-text search, Trigram similarity
INFO:src.core:   - Ready for: 10 search methods


✓ HNSW system initialized


### Add Documents & Build Index

In [7]:
# Add documents
doc_ids = await hnsw_rag.add_documents(documents)
print(f"✓ Added {len(doc_ids)} documents")

# Create metadata indexes for filtering
await hnsw_rag.create_metadata_index(["category", "language", "author"])

# Build HNSW index (m=16, ef_construction=64)
await hnsw_rag.build_index(m=16, ef_construction=64)
print("✓ HNSW index built")

INFO:src.core:Added 10 documents
INFO:src.core:✓ Metadata index created for column: category
INFO:src.core:✓ Metadata index created for column: language
INFO:src.core:✓ Metadata index created for column: author
INFO:src.core:Metadata indexes created for: ['category', 'language', 'author']
INFO:src.core:HNSW index built (m=16, ef_construction=64)
INFO:src.core:hnsw index built successfully


✓ Added 10 documents
✓ HNSW index built


### METHOD 1: Pure Keyword Search

In [8]:
results = await hnsw_rag.keyword_search("machine learning algorithms", k=3)

print("=== METHOD 1: Keyword Search ===")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")

=== METHOD 1: Keyword Search ===

1. Score: 0.2683
   Content: Machine learning algorithms can identify patterns in large datasets automaticall...
   Category: ai


### METHOD 2: Universal Keyword Search (Content + Metadata Fields)

In [9]:
# Search in both content and metadata fields (author, category, etc.)
results = await hnsw_rag.universal_keyword_search(
    query="Python",
    k=4,
    metadata_fields=["author", "category"]  # Also search in these metadata fields
)

print("=== METHOD 2: Universal Keyword Search ===")
print("Searching 'Python' in content AND metadata (author, category)")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Author: {res['metadata'].get('author')}")
    print(f"   Category: {res['metadata'].get('category')}")

=== METHOD 2: Universal Keyword Search ===
Searching 'Python' in content AND metadata (author, category)

1. Score: 0.0608
   Content: Python is a high-level programming language known for its simplicity and readabi...
   Author: Tech Expert
   Category: programming

2. Score: 0.0608
   Content: FastAPI is a modern Python framework for building high-performance APIs quickly....
   Author: Backend Developer
   Category: programming


### METHOD 3: Pure Semantic Search (Vector Similarity)

In [10]:
# Semantic search using vector embeddings (finds conceptually similar content)
results = await hnsw_rag.semantic_search(
    query="understanding human speech",  # Different words, same meaning as "NLP"
    k=3
)

print("=== METHOD 3: Semantic Search ===")
print("Query: 'understanding human speech'")
print("(Should find NLP/language-related docs even without exact keywords)")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Distance: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")

=== METHOD 3: Semantic Search ===
Query: 'understanding human speech'
(Should find NLP/language-related docs even without exact keywords)

1. Distance: 0.5011
   Content: Natural language processing enables computers to understand and generate human l...
   Category: ai

2. Distance: 0.6656
   Content: Transformer models revolutionized NLP with attention mechanisms and parallel pro...
   Category: ai

3. Distance: 0.7271
   Content: Deep learning neural networks can solve complex problems like image recognition....
   Category: ai


### METHOD 4: Pure Metadata Filter (No Query)

In [11]:
# Pure metadata filtering without any text/semantic search
results = await hnsw_rag.metadata_filter(
    filter={
        "$and": [
            {"category": {"$eq": "programming"}},
            {"year": {"$gte": 2023}}
        ]
    },
    k=5,
    order_by="year",
    ascending=False
)

print("=== METHOD 4: Pure Metadata Filter ===")
for i, res in enumerate(results, 1):
    print(f"\n{i}. ID: {res['id']}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")
    print(f"   Year: {res['metadata'].get('year')}")

=== METHOD 4: Pure Metadata Filter ===

1. ID: 14d55ed4-6067-4351-b025-ff9ca6809f71
   Content: Python is a high-level programming language known for its simplicity and readabi...
   Category: programming
   Year: 2024

2. ID: 7286e328-81d8-43e7-9683-a798d808ec5d
   Content: FastAPI is a modern Python framework for building high-performance APIs quickly....
   Category: programming
   Year: 2023


### METHOD 5: Metadata + Keyword Search

In [12]:
# Search for "database" keyword only in 'database' category
results = await hnsw_rag.metadata_keyword_search(
    query="database",
    filter={"category": {"$eq": "database"}},
    k=3
)

print("=== METHOD 5: Metadata + Keyword Search ===")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")

=== METHOD 5: Metadata + Keyword Search ===

1. Score: 0.0608
   Content: PostgreSQL is a powerful open-source relational database management system....
   Category: database

2. Score: 0.0608
   Content: Vector databases store and retrieve data based on semantic similarity using embe...
   Category: database


### METHOD 6: Metadata + Semantic Search

In [13]:
# Semantic search only in 'ai' category
results = await hnsw_rag.metadata_semantic_search(
    query="understanding language",
    filter={"category": {"$eq": "ai"}},
    k=3
)

print("=== METHOD 6: Metadata + Semantic ===")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Distance: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")

=== METHOD 6: Metadata + Semantic ===

1. Distance: 0.5045
   Content: Natural language processing enables computers to understand and generate human l...
   Category: ai

2. Distance: 0.6973
   Content: Transformer models revolutionized NLP with attention mechanisms and parallel pro...
   Category: ai

3. Distance: 0.7419
   Content: Deep learning neural networks can solve complex problems like image recognition....
   Category: ai


### METHOD 7: Hybrid Search (Keyword + Semantic)

In [14]:
# Combine keyword and semantic search with 50/50 weighting
results = await hnsw_rag.hybrid_search(
    query="Python programming frameworks",
    k=3,
    weights=(0.5, 0.5)  # (semantic_weight, keyword_weight)
)

print("=== METHOD 7: Hybrid Search ===")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Fused Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")

=== METHOD 7: Hybrid Search ===

1. Fused Score: 0.5000
   Content: Python is a high-level programming language known for its simplicity and readabi...
   Category: programming

2. Fused Score: 0.3382
   Content: FastAPI is a modern Python framework for building high-performance APIs quickly....
   Category: programming

3. Fused Score: 0.0911
   Content: React is a JavaScript library for building user interfaces with reusable compone...
   Category: web


#### Hybrid Search with RRF Scoring (Alternative)

In [15]:
# Use Reciprocal Rank Fusion (RRF) instead of weighted scoring
# RRF doesn't require manual weight tuning!
results_rrf = await hnsw_rag.hybrid_search(
    query="Python programming frameworks",
    k=3,
    use_rrf=True,  # Enable RRF scoring
    rrf_k=60       # RRF constant (default: 60)
)

print("=== Hybrid Search with RRF Scoring ===")
print("(Reciprocal Rank Fusion - no weight tuning needed)")
for i, res in enumerate(results_rrf, 1):
    print(f"\n{i}. RRF Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}")

=== Hybrid Search with RRF Scoring ===
(Reciprocal Rank Fusion - no weight tuning needed)

1. RRF Score: 0.0164
   Content: Python is a high-level programming language known for its simplicity and readabi...
   Category: programming

2. RRF Score: 0.0161
   Content: FastAPI is a modern Python framework for building high-performance APIs quickly....
   Category: programming

3. RRF Score: 0.0159
   Content: React is a JavaScript library for building user interfaces with reusable compone...
   Category: web


### METHOD 8: Ensemble Search (Metadata + Hybrid)

In [16]:
# Most comprehensive: filter by metadata + hybrid search
results = await hnsw_rag.ensemble_search(
    query="neural networks",
    filter={"year": {"$gte": 2024}},
    k=3,
    weights=(0.6, 0.4)  # Favor semantic over keyword
)

print("=== METHOD 8: Ensemble Search ===")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Fused Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Year: {res['metadata'].get('year')}")

=== METHOD 8: Ensemble Search ===

1. Fused Score: 1.0000
   Content: Deep learning neural networks can solve complex problems like image recognition....
   Year: 2024

2. Fused Score: 0.2918
   Content: Machine learning algorithms can identify patterns in large datasets automaticall...
   Year: 2024

3. Fused Score: 0.2564
   Content: Transformer models revolutionized NLP with attention mechanisms and parallel pro...
   Year: 2024


### METHOD 9: Trigram Search (Fuzzy/Typo-Tolerant)

In [17]:
# Fuzzy text matching - handles typos and spelling variations
results = await hnsw_rag.trigram_search(
    query="artifical inteligence",  # Note the typos!
    k=3,
    threshold=0.3  # Min similarity score (0.0-1.0)
)

print("=== METHOD 9: Trigram Search (Fuzzy Matching) ===")
print("Query with typos: 'artifical inteligence'")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Similarity Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Year: {res['metadata'].get('year')}")

=== METHOD 9: Trigram Search (Fuzzy Matching) ===
Query with typos: 'artifical inteligence'


### METHOD 10: Metadata + Trigram Search (Filtered Fuzzy Search)

In [18]:
# Mandatory metadata filtering + fuzzy text matching
results = await hnsw_rag.metadata_trigram_search(
    query="machne lerning tutrial",  # Multiple typos!
    filter={
        "$and": [
            {"category": "programming"},
            {"year": {"$gte": 2023}}
        ]
    },
    k=3,
    threshold=0.3
)

print("=== METHOD 10: Metadata + Trigram Search ===")
print("Query with typos: 'machne lerning tutrial'")
print("Filter: category=programming AND year>=2023")
for i, res in enumerate(results, 1):
    print(f"\n{i}. Similarity Score: {res['score']:.4f}")
    print(f"   Content: {res['content'][:80]}...")
    print(f"   Category: {res['metadata'].get('category')}, Year: {res['metadata'].get('year')}")

=== METHOD 10: Metadata + Trigram Search ===
Query with typos: 'machne lerning tutrial'
Filter: category=programming AND year>=2023


## 4. Filter Operators Demo (All 13 Types)

### Comparison Operators: $eq, $ne, $lt, $lte, $gt, $gte

In [19]:
print("=== $eq: Equal ===")
results = await hnsw_rag.metadata_semantic_search(
    "technology", {"category": {"$eq": "ai"}}, k=2
)
print(f"Found {len(results)} AI documents\n")

print("=== $ne: Not Equal ===")
results = await hnsw_rag.metadata_semantic_search(
    "technology", {"category": {"$ne": "ai"}}, k=2
)
print(f"Found {len(results)} non-AI documents\n")

print("=== $gte: Greater Than or Equal ===")
results = await hnsw_rag.metadata_semantic_search(
    "recent", {"year": {"$gte": 2024}}, k=3
)
print(f"Found {len(results)} documents from 2024+")
for res in results:
    print(f"  - Year {res['metadata']['year']}: {res['content'][:60]}...")

=== $eq: Equal ===
Found 2 AI documents

=== $ne: Not Equal ===
Found 2 non-AI documents

=== $gte: Greater Than or Equal ===
Found 3 documents from 2024+
  - Year 2024: Machine learning algorithms can identify patterns in large d...
  - Year 2024: Transformer models revolutionized NLP with attention mechani...
  - Year 2024: Vector databases store and retrieve data based on semantic s...


### Set Operators: $in, $nin

In [20]:
print("=== $in: Value In List ===")
results = await hnsw_rag.metadata_semantic_search(
    "coding",
    {"language": {"$in": ["Python", "JavaScript"]}},
    k=4
)
print(f"Found {len(results)} Python/JavaScript documents")
for res in results:
    print(f"  - {res['metadata']['language']}: {res['content'][:60]}...\n")

print("\n=== $nin: Value Not In List ===")
results = await hnsw_rag.metadata_semantic_search(
    "technology",
    {"category": {"$nin": ["ai", "programming"]}},
    k=3
)
print(f"Found {len(results)} documents (not ai/programming)")
for res in results:
    print(f"  - {res['metadata']['category']}: {res['content'][:60]}...")

=== $in: Value In List ===
Found 4 Python/JavaScript documents
  - Python: Natural language processing enables computers to understand ...

  - Python: Python is a high-level programming language known for its si...

  - Python: Deep learning neural networks can solve complex problems lik...

  - Python: Transformer models revolutionized NLP with attention mechani...


=== $nin: Value Not In List ===
Found 3 documents (not ai/programming)
  - web: React is a JavaScript library for building user interfaces w...
  - database: PostgreSQL is a powerful open-source relational database man...
  - devops: Docker containers provide isolated environments for running ...


### Range Operator: $between

In [21]:
print("=== $between: Range Query ===")
results = await hnsw_rag.metadata_semantic_search(
    "technology",
    {"year": {"$between": [2023, 2024]}},
    k=5
)
print(f"Found {len(results)} documents from 2023-2024")
for res in results:
    print(f"  - Year {res['metadata']['year']}: {res['content'][:60]}...")

=== $between: Range Query ===
Found 5 documents from 2023-2024
  - Year 2024: Natural language processing enables computers to understand ...
  - Year 2024: Deep learning neural networks can solve complex problems lik...
  - Year 2023: React is a JavaScript library for building user interfaces w...
  - Year 2024: Python is a high-level programming language known for its si...
  - Year 2024: Machine learning algorithms can identify patterns in large d...


### Existence Operator: $exists

In [22]:
print("=== $exists: Field Presence Check ===")
results = await hnsw_rag.metadata_semantic_search(
    "content",
    {"author": {"$exists": True}},
    k=3
)
print(f"Found {len(results)} documents with 'author' field")
for res in results:
    print(f"  - Author: {res['metadata']['author']}")

=== $exists: Field Presence Check ===
Found 3 documents with 'author' field
  - Author: NLP Specialist
  - Author: DB Expert
  - Author: Data Engineer


### Pattern Operators: $like, $ilike

In [23]:
print("=== $like: Case-Sensitive Pattern ===")
results = await hnsw_rag.metadata_semantic_search(
    "expert",
    {"author": {"$like": "%Expert"}},
    k=3
)
print(f"Found {len(results)} documents by 'Expert' authors")
for res in results:
    print(f"  - {res['metadata']['author']}: {res['content'][:60]}...\n")

print("\n=== $ilike: Case-Insensitive Pattern ===")
results = await hnsw_rag.metadata_semantic_search(
    "developer",
    {"author": {"$ilike": "%developer%"}},
    k=3
)
print(f"Found {len(results)} documents by 'developer' (any case)")
for res in results:
    print(f"  - {res['metadata']['author']}: {res['content'][:60]}...")

=== $like: Case-Sensitive Pattern ===
Found 3 documents by 'Expert' authors
  - Deep Learning Expert: Deep learning neural networks can solve complex problems lik...

  - Tech Expert: Python is a high-level programming language known for its si...

  - DB Expert: PostgreSQL is a powerful open-source relational database man...


=== $ilike: Case-Insensitive Pattern ===
Found 2 documents by 'developer' (any case)
  - Frontend Developer: React is a JavaScript library for building user interfaces w...
  - Backend Developer: FastAPI is a modern Python framework for building high-perfo...


### Logical Operators: $and, $or

In [24]:
print("=== $and: Multiple Conditions (ALL must match) ===")
results = await hnsw_rag.metadata_semantic_search(
    "AI technology",
    {
        "$and": [
            {"category": {"$eq": "ai"}},
            {"year": {"$gte": 2024}},
            {"language": {"$eq": "Python"}}
        ]
    },
    k=3
)
print(f"Found {len(results)} AI + 2024+ + Python documents")
for res in results:
    print(f"  - {res['metadata']['category']}, {res['metadata']['year']}, {res['metadata']['language']}")
    print(f"    {res['content'][:60]}...\n")

print("\n=== $or: Multiple Conditions (ANY can match) ===")
results = await hnsw_rag.metadata_semantic_search(
    "technology",
    {
        "$or": [
            {"category": {"$eq": "web"}},
            {"category": {"$eq": "devops"}}
        ]
    },
    k=3
)
print(f"Found {len(results)} web OR devops documents")
for res in results:
    print(f"  - {res['metadata']['category']}: {res['content'][:60]}...")

=== $and: Multiple Conditions (ALL must match) ===
Found 3 AI + 2024+ + Python documents
  - ai, 2024, Python
    Deep learning neural networks can solve complex problems lik...

  - ai, 2024, Python
    Natural language processing enables computers to understand ...

  - ai, 2024, Python
    Machine learning algorithms can identify patterns in large d...


=== $or: Multiple Conditions (ANY can match) ===
Found 2 web OR devops documents
  - web: React is a JavaScript library for building user interfaces w...
  - devops: Docker containers provide isolated environments for running ...


### Complex Nested Filters

In [25]:
print("=== Complex Nested Filter ===")
# Find: (AI or database) AND Python AND 2024+
results = await hnsw_rag.metadata_semantic_search(
    "advanced technology",
    {
        "$and": [
            {
                "$or": [
                    {"category": {"$eq": "ai"}},
                    {"category": {"$eq": "database"}}
                ]
            },
            {"language": {"$eq": "Python"}},
            {"year": {"$gte": 2024}}
        ]
    },
    k=5
)
print(f"Found {len(results)} documents matching complex filter")
for res in results:
    print(f"  - {res['metadata']['category']}, {res['metadata']['language']}, Year: {res['metadata']['year']}")
    print(f"    {res['content'][:70]}...\n")

=== Complex Nested Filter ===
Found 5 documents matching complex filter
  - ai, Python, Year: 2024
    Natural language processing enables computers to understand and genera...

  - ai, Python, Year: 2024
    Deep learning neural networks can solve complex problems like image re...

  - ai, Python, Year: 2024
    Transformer models revolutionized NLP with attention mechanisms and pa...

  - ai, Python, Year: 2024
    Machine learning algorithms can identify patterns in large datasets au...

  - database, Python, Year: 2024
    Vector databases store and retrieve data based on semantic similarity ...



## 5. IVFFlat Index Demo (Balanced Performance)

In [26]:
# Create RAG system with IVFFlat index
ivf_rag = pgVectorDB(
    collection_name="ivfflat_prod_demo",
    embedding_model=embedding_model,
    connection_string=connection_string,
    index_type=IndexType.IVFFLAT
)

await ivf_rag.initialize(overwrite_existing=True)
await ivf_rag.add_documents(documents)
await ivf_rag.create_metadata_index(["category", "language"])

# Build IVFFlat index (lists will be auto-calculated)
await ivf_rag.build_index(lists=10)
print("✓ IVFFlat system ready")

# Set query parameters (probes=3)
await ivf_rag.set_query_params(probes=3)

# Test semantic search
results = await ivf_rag.semantic_search("machine learning AI", k=3)
print("\n=== IVFFlat Semantic Search ===")
for i, res in enumerate(results, 1):
    print(f"{i}. {res['content'][:70]}... (distance={res['score']:.4f})")

INFO:src.core:pgVectorDB initialized: 'ivfflat_prod_demo' with ivfflat (vector_size=384)
INFO:src.core:Step 1/5: Ensuring PostgreSQL extensions...
INFO:src.core:✓ Extension 'vector' enabled
INFO:src.core:✓ Extension 'pg_trgm' enabled
INFO:src.core:Step 2/5: Creating table...
INFO:src.core:✓ Table 'ivfflat_prod_demo' created
INFO:src.core:Step 3/5: Setting up search indexes...
INFO:src.core:✓ Full-text search index created
INFO:src.core:✓ Trigram similarity index created
INFO:src.core:Step 4/5: Initializing vector store...
INFO:src.core:✓ Vector store initialized
INFO:src.core:✓ Step 5/5: System ready with ivfflat index (vector_size=384)
INFO:src.core:================================================================================
INFO:src.core:🚀 Production RAG System initialized successfully!
INFO:src.core:   - Extensions: vector, pg_trgm
INFO:src.core:   - Table: public.ivfflat_prod_demo
INFO:src.core:   - Indexes: Full-text search, Trigram similarity
INFO:src.core:   - Ready for: 10 

✓ IVFFlat system ready

=== IVFFlat Semantic Search ===
1. Machine learning algorithms can identify patterns in large datasets au... (distance=0.4619)
2. Deep learning neural networks can solve complex problems like image re... (distance=0.5452)
3. Natural language processing enables computers to understand and genera... (distance=0.6747)


## 6. DiskANN Index Demo (Scalable + Label Filtering)

### Initialize DiskANN System

In [31]:
# Create RAG system with DiskANN index
diskann_rag = pgVectorDB(
    collection_name="diskann_prod_demo",
    embedding_model=embedding_model,
    connection_string=connection_string,
    index_type=IndexType.DISKANN
)

await diskann_rag.initialize(overwrite_existing=True)
print("✓ DiskANN system initialized (pgvectorscale required)")

INFO:src.core:pgVectorDB initialized: 'diskann_prod_demo' with diskann (vector_size=384)
INFO:src.core:Step 1/5: Ensuring PostgreSQL extensions...
INFO:src.core:✓ Extension 'vector' enabled
INFO:src.core:✓ Extension 'pg_trgm' enabled
INFO:src.core:✓ Extension 'vectorscale' enabled
INFO:src.core:Step 2/5: Creating table...
INFO:src.core:✓ Table 'diskann_prod_demo' created
INFO:src.core:Step 3/5: Setting up search indexes...
INFO:src.core:✓ Full-text search index created
INFO:src.core:✓ Trigram similarity index created
INFO:src.core:Step 4/5: Initializing vector store...
INFO:src.core:✓ Vector store initialized
INFO:src.core:✓ Step 5/5: System ready with diskann index (vector_size=384)
INFO:src.core:================================================================================
INFO:src.core:🚀 Production RAG System initialized successfully!
INFO:src.core:   - Extensions: vector, pg_trgm, vectorscale
INFO:src.core:   - Table: public.diskann_prod_demo
INFO:src.core:   - Indexes: Full-text

✓ DiskANN system initialized (pgvectorscale required)


### Add Documents with Labels

In [32]:
# Add documents with labels for filtering
await diskann_rag.add_documents(documents, labels=document_labels)
await diskann_rag.create_metadata_index(["category", "language"])

print("✓ Documents added with labels")

INFO:src.core:Labels added for DiskANN filtering
INFO:src.core:Added 10 documents
INFO:src.core:✓ Metadata index created for column: category
INFO:src.core:✓ Metadata index created for column: language
INFO:src.core:Metadata indexes created for: ['category', 'language']


✓ Documents added with labels


### Build DiskANN Index with Labels

In [33]:
# Build DiskANN index with label filtering support
await diskann_rag.build_index(
    num_neighbors=50,
    search_list_size=100,
    storage_layout=StorageLayout.MEMORY_OPTIMIZED,
    include_labels=True
)

# Set query parameters
await diskann_rag.set_query_params(
    query_search_list_size=100,
    query_rescore=50
)

print("✓ DiskANN index built with label support")

INFO:src.core:DiskANN index built (neighbors=50, search_list=100, storage=memory_optimized, labels=True)
INFO:src.core:diskann index built successfully
INFO:src.core:DiskANN: search_list=100, rescore=50


✓ DiskANN index built with label support


### Label-Based Filtering Examples

In [34]:
print("=== Search Only AI Documents (label=2) ===")
results = await diskann_rag.semantic_search(
    "neural networks and learning",
    k=3,
    label_filter=[2]  # Only AI category
)
for i, res in enumerate(results, 1):
    print(f"{i}. {res['metadata']['category']}: {res['content'][:70]}...")

print("\n=== Search AI + Database (labels=2,3) ===")
results = await diskann_rag.semantic_search(
    "data and algorithms",
    k=4,
    label_filter=[2, 3]  # AI + Database
)
for i, res in enumerate(results, 1):
    print(f"{i}. {res['metadata']['category']}: {res['content'][:70]}...")

print("\n=== Search Programming + Web (labels=1,4) ===")
results = await diskann_rag.semantic_search(
    "building applications",
    k=3,
    label_filter=[1, 4]  # Programming + Web
)
for i, res in enumerate(results, 1):
    print(f"{i}. {res['metadata']['category']}: {res['content'][:70]}...")

=== Search Only AI Documents (label=2) ===
1. ai: Deep learning neural networks can solve complex problems like image re...
2. ai: Machine learning algorithms can identify patterns in large datasets au...
3. ai: Natural language processing enables computers to understand and genera...

=== Search AI + Database (labels=2,3) ===
1. ai: Machine learning algorithms can identify patterns in large datasets au...
2. ai: Deep learning neural networks can solve complex problems like image re...
3. database: Vector databases store and retrieve data based on semantic similarity ...
4. database: PostgreSQL is a powerful open-source relational database management sy...

=== Search Programming + Web (labels=1,4) ===
1. web: React is a JavaScript library for building user interfaces with reusab...
2. programming: FastAPI is a modern Python framework for building high-performance API...
3. programming: Python is a high-level programming language known for its simplicity a...


## 7. Performance Comparison

In [35]:
import time

async def benchmark_search(rag_system, query, label_filter=None):
    """Benchmark search performance."""
    times = []
    for _ in range(5):  # Run 5 times
        start = time.time()
        await rag_system.semantic_search(query, k=3, label_filter=label_filter)
        times.append((time.time() - start) * 1000)  # ms
    return sum(times) / len(times)

# Benchmark all three indexes
query = "machine learning and artificial intelligence"

hnsw_time = await benchmark_search(hnsw_rag, query)
ivf_time = await benchmark_search(ivf_rag, query)
diskann_time = await benchmark_search(diskann_rag, query, label_filter=[2])  # With label filter

print("=== Performance Comparison (5 runs average) ===")
print(f"HNSW:    {hnsw_time:.2f} ms")
print(f"IVFFlat: {ivf_time:.2f} ms")
print(f"DiskANN: {diskann_time:.2f} ms (with label filter)")
print("\nNote: HNSW typically fastest but memory-intensive")
print("      DiskANN best for large-scale (>10M vectors) with label filtering")

=== Performance Comparison (5 runs average) ===
HNSW:    25.70 ms
IVFFlat: 29.96 ms
DiskANN: 16.96 ms (with label filter)

Note: HNSW typically fastest but memory-intensive
      DiskANN best for large-scale (>10M vectors) with label filtering


## 8. Error Handling Demonstrations

In [38]:
from src.core import ValidationError, InitializationError, DatabaseError

print("=== Testing Error Handling ===")

# Test 1: Invalid query
try:
    await hnsw_rag.semantic_search("", k=3)
except ValidationError as e:
    print(f"✓ Caught ValidationError: {e}")

# Test 2: Invalid k parameter
try:
    await hnsw_rag.keyword_search("test", k=0)
except ValidationError as e:
    print(f"✓ Caught ValidationError: {e}")

# Test 3: Invalid weights
try:
    await hnsw_rag.hybrid_search("test", k=3, weights=(0.3, 0.5))  # Don't sum to 1.0
except ValidationError as e:
    print(f"✓ Caught ValidationError: {e}")

# Test 4: Invalid filter operator
try:
    await hnsw_rag.metadata_semantic_search(
        "test",
        {"category": {"$invalid": "value"}},
        k=3
    )
except (ValidationError, DatabaseError) as e:
    print(f"✓ Caught error for invalid operator: {str(e).split(':')[-1].strip()}")

print("\n✓ Error handling working correctly!")

=== Testing Error Handling ===
✓ Caught ValidationError: query must be a non-empty string
✓ Caught ValidationError: k must be positive
✓ Caught ValidationError: weights must sum to 1.0, got 0.8
✓ Caught error for invalid operator: $invalid

✓ Error handling working correctly!


## 9. System Statistics

In [36]:
# Get statistics for each system
print("=== HNSW System Stats ===")
hnsw_stats = await hnsw_rag.get_stats()
for key, value in hnsw_stats.items():
    if key != "indexes":
        print(f"{key}: {value}")

print("\n=== IVFFlat System Stats ===")
ivf_stats = await ivf_rag.get_stats()
for key, value in ivf_stats.items():
    if key != "indexes":
        print(f"{key}: {value}")

print("\n=== DiskANN System Stats ===")
diskann_stats = await diskann_rag.get_stats()
for key, value in diskann_stats.items():
    if key != "indexes":
        print(f"{key}: {value}")

=== HNSW System Stats ===
index_type: hnsw
table_name: hnsw_prod_demo
schema_name: public
vector_size: 384
index_built: True
document_count: 10
table_size: 208 kB

=== IVFFlat System Stats ===
index_type: ivfflat
table_name: ivfflat_prod_demo
schema_name: public
vector_size: 384
index_built: True
document_count: 10
table_size: 264 kB

=== DiskANN System Stats ===
index_type: diskann
table_name: diskann_prod_demo
schema_name: public
vector_size: 384
index_built: True
document_count: 10
table_size: 216 kB


## 10. Cleanup

In [37]:
# Close all connections
await hnsw_rag.close()
await ivf_rag.close()
await diskann_rag.close()

print("✓ All systems closed successfully")

INFO:src.core:Database connections closed
INFO:src.core:Database connections closed
INFO:src.core:Database connections closed


✓ All systems closed successfully


## Summary

### Key Features Demonstrated:

**Index Types:**
- **HNSW**: Fast in-memory index, best for <1M vectors
- **IVFFlat**: Balanced performance, configurable probes
- **DiskANN**: Scalable disk-based index with label filtering

**Search Methods:**
1. **Keyword Search**: Pure full-text search
2. **Semantic Search**: Vector similarity
3. **Metadata + Keyword**: Filtered full-text
4. **Metadata + Semantic**: Filtered vector search
5. **Hybrid Search**: Combined keyword + semantic
6. **Ensemble Search**: Metadata + hybrid

**13 Filter Operators:**
- Comparison: `$eq`, `$ne`, `$lt`, `$lte`, `$gt`, `$gte`
- Set: `$in`, `$nin`
- Range: `$between`
- Existence: `$exists`
- Pattern: `$like`, `$ilike`
- Logical: `$and`, `$or`

**Additional Features:**
- Connection pooling for production deployments
- Custom exception hierarchy
- Input validation
- Label-based filtering (DiskANN)
- Query parameter tuning
- Comprehensive error handling

### When to Use Each Index:
- **HNSW**: Highest recall, fastest queries, limited by RAM (~1M vectors)
- **IVFFlat**: Good balance, tune with probes parameter (100K-10M vectors)
- **DiskANN**: Best for large-scale (>10M), disk-based, label filtering support